# OpenSOC-AI — Cell 1: Install Dependencies

**Runtime: T4 GPU required** → Runtime → Change runtime type → T4 GPU

Run this cell once, then **restart the runtime** before proceeding.

In [ ]:
# ============================================================
# CELL 1: Install Dependencies
# After this cell completes: Runtime → Restart session
# ============================================================

!pip uninstall -q -y bitsandbytes triton peft trl transformers accelerate 2>/dev/null
!pip install -q 'bitsandbytes>=0.43.3'
!pip install -q \
    transformers==4.40.2 \
    peft==0.10.0 \
    trl==0.8.6 \
    accelerate==0.30.1 \
    datasets==2.19.0 \
    sentencepiece \
    protobuf

import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — check runtime type"}')
print('\n✅ Done. RESTART RUNTIME now, then run Cell 2.')

---
# Cell 2: Mount Drive + Verify Files

Run after restarting runtime. Fix any missing file paths before continuing.

In [ ]:
# ============================================================
# CELL 2: Mount Google Drive + Verify Files
# ============================================================

from google.colab import drive
import os, json

drive.mount('/content/drive')

DRIVE_ROOT   = '/content/drive/MyDrive'
TRAIN_PATH   = f'{DRIVE_ROOT}/soc_train.json'
EVAL_PATH    = f'{DRIVE_ROOT}/soc_eval.json'
ADAPTER_SAVE = f'{DRIVE_ROOT}/opensoc-adapters'
EVAL_OUT     = f'{DRIVE_ROOT}/eval_results.json'

print('Checking required files...\n')
all_ok = True
for name, path in [('soc_train.json', TRAIN_PATH), ('soc_eval.json', EVAL_PATH)]:
    exists = os.path.exists(path)
    print(f'  {"✅" if exists else "❌ MISSING"}  {name}  →  {path}')
    if exists:
        with open(path) as f: data = json.load(f)
        print(f'         └─ {len(data)} examples')
    else:
        all_ok = False

print()
print('✅ Safe to run Cell 3.' if all_ok else '❌ Upload missing files to Google Drive root and re-run.')

---
# Cell 3: Fine-tune + Save LoRA Adapters

⚠️ **Critical rules:**
- ✅ Saves ONLY `adapter_config.json` + `adapter_model.safetensors`
- ❌ NEVER calls `merge_and_unload()` — corrupts 4-bit weights
- ❌ NEVER calls `model.save_pretrained()` on the full model

In [ ]:
# ============================================================
# CELL 3: Fine-tune TinyLlama-1.1B with LoRA
# Saves ONLY adapter weights to Google Drive
# ============================================================

import os, json, torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer

DRIVE_ROOT   = '/content/drive/MyDrive'
TRAIN_PATH   = f'{DRIVE_ROOT}/soc_train.json'
ADAPTER_SAVE = f'{DRIVE_ROOT}/opensoc-adapters'
BASE_MODEL   = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

# 1. Dataset
def format_example(ex):
    return {'text': f"### Instruction:\n{ex['instruction']}\n\n### Input:\n{ex['input']}\n\n### Response:\n{ex['output']}"}

with open(TRAIN_PATH) as f: raw = json.load(f)
dataset = Dataset.from_list(raw).map(format_example)
print(f'✅ Dataset: {len(dataset)} examples')

# 2. Load base model (4-bit QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map='auto')
model.config.use_cache = False
print('✅ Base model loaded (4-bit)')

# 3. LoRA config
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05, bias='none', task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# 4. Train
training_args = TrainingArguments(
    output_dir='/content/checkpoints', num_train_epochs=3,
    per_device_train_batch_size=4, gradient_accumulation_steps=4,
    learning_rate=2e-4, fp16=True, logging_steps=25, save_strategy='no',
    warmup_ratio=0.05, lr_scheduler_type='cosine',
    optim='paged_adamw_8bit', report_to='none',
)
trainer = SFTTrainer(
    model=model, train_dataset=dataset, dataset_text_field='text',
    max_seq_length=512, tokenizer=tokenizer, args=training_args, packing=False,
)
print('🚀 Training...')
trainer.train()
print('✅ Training complete!')

# 5. Save ONLY LoRA adapters — ⚠️ DO NOT merge_and_unload()
os.makedirs(ADAPTER_SAVE, exist_ok=True)
trainer.model.save_pretrained(ADAPTER_SAVE)
tokenizer.save_pretrained(ADAPTER_SAVE)

saved = os.listdir(ADAPTER_SAVE)
print(f'\n✅ Adapters saved to: {ADAPTER_SAVE}')
print(f'   Files: {saved}')
assert 'adapter_config.json' in saved
assert any('adapter_model' in f for f in saved)
print('🎉 Verified. Adapters only — full model NOT saved.')